# Validation Test Execution Notebook
### Air Quality Review (AQR) System — GAMP 5 Category 5 Validation Protocol

This notebook contains individual cells to execute and verify every test case in the combined validation protocol (Appendix 2 and 3).
All test data is prepared by copying and programmatically editing real files from the project's `data/` directory to ensure high-fidelity testing without mock templates.

## Setup & Helpers

In [1]:
import sys
import os
import shutil
import pandas as pd
import numpy as np
from datetime import datetime

# Ensure project root is in system path
project_dir = r'D:\ex_work\AirQualityReview_Project'
if project_dir not in sys.path:
    sys.path.append(project_dir)

import analysis_logic
import audit_trail

# Setup test data workspace directory
test_workspace = os.path.join(project_dir, 'data', 'validation_tests')
os.makedirs(test_workspace, exist_ok=True)
print("Validation test workspace configured at:", test_workspace)

Validation test workspace configured at: D:\ex_work\AirQualityReview_Project\data\validation_tests


# Part 1: Module & Unit Verification (Appendix 2)

### MT-01: Software Installation & Env Verification

In [2]:
print("--- Executing MT-01: Environment Initialization & Self-Healing ---")

# 1. Verify existence of build files
exe_info_path = os.path.join(project_dir, 'app_version_info.txt')
if os.path.exists(exe_info_path):
    with open(exe_info_path, 'r') as f:
        print("App Version Info Contents:\n", f.read().strip())

# 2. Test self-healing: Clear logs and reports contents to verify they are recreated/usable
temp_reports = os.path.join(project_dir, 'reports')
temp_logs = os.path.join(project_dir, 'logs')

for path in [temp_reports, temp_logs]:
    if os.path.exists(path):
        for f in os.listdir(path):
            f_path = os.path.join(path, f)
            try:
                if os.path.isfile(f_path): os.remove(f_path)
            except Exception as e: print(f"Skipping file {f}: {e}")

print("Cleared logs and reports folder contents to test self-healing.")

# Trigger recreations
os.makedirs(temp_reports, exist_ok=True)
os.makedirs(temp_logs, exist_ok=True)
audit_trail.log_event("TEST_SELF_HEAL", "Folders successfully self-healed")

print("Is reports folder present?:", os.path.exists(temp_reports))
print("Is logs folder present?:", os.path.exists(temp_logs))
print("Is audit_trail.log created?:", os.path.exists(os.path.join(temp_logs, 'audit_trail.log')))

# .venv\Scripts\Activate.ps1
# pyinstaller AQR_Dashboard_v1.1.0_Fix.spec

--- Executing MT-01: Environment Initialization & Self-Healing ---
App Version Info Contents:
 # IQ-TC-01: Binary Integrity & Versioning Verification
#
# For more details about fixed file info:
# http://msdn.microsoft.com/en-us/library/ms646997.aspx
VSVersionInfo(
  ffi=FixedFileInfo(
    filevers=(1, 1, 0, 0),
    prodvers=(1, 1, 0, 0),
    mask=0x3f,
    flags=0x0,
    OS=0x40004,
    fileType=0x1,
    subtype=0x0,
    date=(0, 0)
    ),
  kids=[
    StringFileInfo(
      [
      StringTable(
        u'040904B0',
        [StringStruct(u'CompanyName', u'AQR Program'),
        StringStruct(u'FileDescription', u'Air Quality Review System - GAMP 5 Compliant'),
        StringStruct(u'FileVersion', u'1.1.0'),
        StringStruct(u'InternalName', u'AirQualityReview'),
        StringStruct(u'LegalCopyright', u'Â© 2026 AQR Program. All rights reserved.'),
        StringStruct(u'OriginalFilename', u'AirQualityReview_1.1.0.exe'),
        StringStruct(u'ProductName', u'Air Quality Review'),
   

### MT-02: Cryptographic Hashing (`get_file_hash`)

In [3]:
print("--- Executing MT-02: File Hash Traceability ---")

# Setup a temporary file
test_file = os.path.join(test_workspace, 'mt02_test.txt')
with open(test_file, 'w') as f:
    f.write("Valid calibration data context.")

# Calculate hash
hash_orig = analysis_logic.get_file_hash(test_file)
print("Original File Hash:", hash_orig)
assert len(hash_orig) == 64, "Hash must be 64 characters hexadecimal"

# Modify file
with open(test_file, 'w') as f:
    f.write("Tampered calibration data context.")
hash_mod = analysis_logic.get_file_hash(test_file)
print("Modified File Hash:", hash_mod)
assert hash_orig != hash_mod, "Hash must change upon file modification"

# Test invalid file path
err_hash = analysis_logic.get_file_hash(os.path.join(test_workspace, 'non_existent.txt'))
print("Invalid File Hash Response:", err_hash)
assert err_hash.startswith("ERROR:"), "Must return an error message string"


--- Executing MT-02: File Hash Traceability ---
Original File Hash: 3852bbdd4e7173492eb07202e62c8622a2435d51f456fe7512cf58b57154f25c
Modified File Hash: f462b990918f748df0654e7d80b6b358df8803b919362213784cffd73459902a
Invalid File Hash Response: ERROR: [Errno 2] No such file or directory: 'D:\\ex_work\\AirQualityReview_Project\\data\\validation_tests\\non_existent.txt'


### MT-03: Header Row Detection (`find_header`)

In [4]:
print("--- Executing MT-03: find_header ---")

# Header at index 0
lines_1 = ["<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_1 = analysis_logic.find_header(lines_1)
print("Header at beginning row index:", idx_1)
assert idx_1 == 0

# Header after multiple rows
lines_2 = ["BAS System Log File", "Generated on Friday", "<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_2 = analysis_logic.find_header(lines_2)
print("Header after multiple rows index:", idx_2)
assert idx_2 == 2

# Missing header
lines_3 = ["Wrong header row name", "Time,Value"]
idx_3 = analysis_logic.find_header(lines_3)
print("Missing header index response:", idx_3)
assert idx_3 is None

print('---------------')
for i in lines_1:
    print(i)
print('---------------')
for i in lines_1:
    print(i)

--- Executing MT-03: find_header ---
Header at beginning row index: 0
Header after multiple rows index: 2
Missing header index response: None
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5


### MT-04: Dynamic Room Point Mapping (`find_point_mapping`)

In [5]:
print("--- Executing MT-04: find_point_mapping ---")

headers = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1-P040 ROOM TEMP","","5 minutes"',
    '"Point_2:","1-P040 ROOM .RMH","","5 minutes"',
    '"Point_3:","1-P040 ROOM PRES","","5 minutes"',

]

headers_2 = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1-P040 ROOM PRES","","5 minutes"',
]

print('-----------------')
pt_temp = analysis_logic.find_point_mapping(headers, "1-P040", "TEMP")
print("Mapped temperature point:", pt_temp)

pt_temp = analysis_logic.find_point_mapping(headers, "1-P040", ".RMH")
print("Mapped humidity point:", pt_temp)

pt_temp = analysis_logic.find_point_mapping(headers, "1-P040", "HUM")
print("Mapped pressure point:", pt_temp)

print('-----------------')
pt_temp = analysis_logic.find_point_mapping(headers_2, "1P040", "PRES")
print("Mapped temperature point:", pt_temp)

print('-----------------')
pt_temp = analysis_logic.find_point_mapping("TEMP", "1P040", "PRES")
print("Mapped temperature point:", pt_temp)

# print('-----------------')
# pt_temp = analysis_logic.find_point_mapping()
# print("Mapped temperature point:", pt_temp)

--- Executing MT-04: find_point_mapping ---
-----------------
Mapped temperature point: Point_1
Mapped humidity point: Point_2
Mapped pressure point: Point_2
-----------------
Mapped temperature point: None
-----------------
Mapped temperature point: None


### MT-05: Continuous Index Sequence Delineation (`find_continuous_ranges`)

In [6]:
print("--- Executing MT-05: find_continuous_ranges ---")

# Sequential values
seq = [1, 2, 3, 5, 6, 10, 11, 13]
seq_2 = [1, 3, 5, 7, 9, 11]
seq_3 = []
ranges = analysis_logic.find_continuous_ranges(seq, min_length=2)
print("Continuous ranges:", ranges)
assert ranges == [(1, 3), (5, 6), (10, 11)]

ranges_2 = analysis_logic.find_continuous_ranges(seq_2, min_length=2)
print("Un-continuous ranges:", ranges_2)

# Empty list
ranges_3 = analysis_logic.find_continuous_ranges(seq_3, min_length=2)
print("Empty ranges:", ranges_3)

--- Executing MT-05: find_continuous_ranges ---
Continuous ranges: [(1, 3), (5, 6), (10, 11)]
Un-continuous ranges: []
Empty ranges: []


### MT-06: Date Bound Extraction (`get_file_date_range`)

In [7]:
print("--- Executing MT-06: get_file_date_range ---")

# Copy standard file and check
src_csv = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
dest_csv = os.path.join(test_workspace, 'mt06_valid.csv')
# shutil.copyfile(src_csv, dest_csv)

start_d, end_d = analysis_logic.get_file_date_range(src_csv)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(dest_csv)
print(f"Parsed valid range: {start_d} to {end_d}")

# Verify with outlier dates (outside 2001-2099)
outlier_csv = os.path.join(test_workspace, 'mt06_outlier.csv')
with open(outlier_csv, 'w') as f:
    f.write('"<>Date","Time"\\n"01/01/1990","00:00"\\n"12/31/2100","23:55"')
os_d, oe_d = analysis_logic.get_file_date_range(outlier_csv)
print(f"Outlier range parsed: {os_d} to {oe_d}")


--- Executing MT-06: get_file_date_range ---
Parsed valid range: 2026-05-13 to 2026-05-13
Parsed valid range: None to None
Outlier range parsed: None to None


### MT-07: Standardized DataFrame Preparation (`prepare_df`)

In [8]:
print("--- Executing MT-07: prepare_df ---")

src_csv = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path)

df = analysis_logic.prepare_df(src_csv, "1-P040", setpoint_df)
print("Reindexed DataFrame Columns:", df.columns.tolist())
print("Types in DataFrame Temperature:", df['Temperature'].dtype)
print("First 5 rows:\n", df.head(10))
assert 'Temperature' in df.columns
assert 'Humidity' in df.columns
assert 'Pressure' in df.columns


--- Executing MT-07: prepare_df ---
Reindexed DataFrame Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Types in DataFrame Temperature: float64
First 5 rows:
 0            DateTime  Temperature  Humidity  Pressure
0 2026-05-13 00:00:00         21.5      41.3      12.9
1 2026-05-13 00:05:00         21.5      41.5      12.9
2 2026-05-13 00:10:00         21.5      41.6      13.0
3 2026-05-13 00:15:00         21.6      41.6      12.9
4 2026-05-13 00:20:00         21.6      41.8      12.9
5 2026-05-13 00:25:00         21.7      41.9      13.0
6 2026-05-13 00:30:00         21.7      41.9      13.0
7 2026-05-13 00:35:00         21.7      41.9      12.9
8 2026-05-13 00:40:00         21.7      42.0      12.9
9 2026-05-13 00:45:00         21.7      42.0      12.9


### MT-08: Pressure Corridor Mapping (`find_compare_path`)

In [9]:
print("--- Executing MT-08: find_compare_path ---")

limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path).dropna(subset=['Room_number'])

file_list = [
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv",
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P999_01-05-26_01-00.csv"
]

# Room 1-P040 comparison corridor is 1-P051
comp_room, comp_path = analysis_logic.find_compare_path(file_list, setpoint_df, "1-P040")
print(f"Room 1-P040 Comparison Room: {comp_room}")
print(f"Comparison File Path: {comp_path}")
# assert comp_room == "1-P051"
# assert comp_path is not None


--- Executing MT-08: find_compare_path ---
Room 1-P040 Comparison Room: 1-P051
Comparison File Path: D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv


### MT-09: Legacy Date Parsing & Offset (`parse_filename_for_datetime`)

In [10]:
print("--- Executing MT-09: parse_filename_for_datetime ---")

filename = "1-P040_05-30-26_00-00.csv"
parsed_date = analysis_logic.parse_filename_for_datetime(filename)
print("Filename: ", filename)
print("Parsed previous-day business date: ", parsed_date)
# 05-30-26 parsed is 2026-05-30. Previous day rule makes it 2026-05-29
assert parsed_date == datetime(2026, 5, 29).date()


--- Executing MT-09: parse_filename_for_datetime ---
Filename:  1-P040_05-30-26_00-00.csv
Parsed previous-day business date:  2026-05-29


### MT-10: Phase II Unified Data Generation (`prepare_df_phase2`)

In [11]:
print("--- Executing MT-10: prepare_df_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b11\2026-05-01"
limit_path_p2 = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit_B11.xlsx"
setpoint_df_p2 = pd.read_excel(limit_path_p2)

room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(phase2_room_dir, "11-1-012", setpoint_df_p2)
print("Unified Room ID:", room_id)
print("Merged Columns:", df_p2.columns.tolist())
print("Discovered Sensors:", sensors)
display(df_p2.head())
assert 'Temperature' in df_p2.columns
assert 'Humidity' in df_p2.columns
assert 'Pressure' in df_p2.columns


--- Executing MT-10: prepare_df_phase2 ---
Unified Room ID: 11-1-012
Merged Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Discovered Sensors: {'Temperature', 'Pressure', 'Humidity'}


,DateTime,Temperature,Humidity,Pressure
0,2026-04-30 08:00:00,22.47,49.04,15.46
1,2026-04-30 08:05:00,22.44,48.53,14.12
2,2026-04-30 08:10:00,22.44,48.53,16.90
3,2026-04-30 08:15:00,22.40,48.79,15.92
4,2026-04-30 08:20:00,22.40,48.79,15.94


### MT-11: Phase II Multiple Files Date Extraction (`get_file_date_range_phase2`)

In [12]:
print("--- Executing MT-11: get_file_date_range_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b11\2026-05-01"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "11-1-012")
print(f"Phase II date range: {s_date} to {e_date}")
assert s_date is not None and e_date is not None


--- Executing MT-11: get_file_date_range_phase2 ---
Phase II date range: 2026-04-30 to 2026-05-01


# Part 2: Integration, System Transformation & Error Verification (Appendix 3)

### ERR-001: Missing Header Row Detection

In [13]:
print("--- Executing ERR-001: Missing header check ---")

err_dir = os.path.join(test_workspace, 'case_err001')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file missing <>Date
with open(err_file, 'w') as f:
    f.write('BAS Log file\nWrongHeaderRow,Time,Point_1\n05/30/2026,00:00,22.5\n')

try:
    analysis_logic.prepare_df(err_file, "1-P040")
    print("FAIL: Expected error was not raised")
except ValueError as e:
    print("PASS: Expected error message caught:", e)
    assert "ERR-001" in str(e)


--- Executing ERR-001: Missing header check ---
PASS: Expected error message caught: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err001\1-P040_05-30-26_00-00.csv


### ERR-002: Missing Limit File Detection

In [14]:
print("--- Executing ERR-002: Missing limit file ---")

folder_path = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C"
missing_limit_path = r"D:\ex_work\AirQualityReview_Project\data\MissingLimit.xlsx"

out, logs, plot = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=missing_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Analysis return output path:", out)
print("Log output:\n", logs)
assert out is None
assert "ERR-002" in logs


--- Executing ERR-002: Missing limit file ---
Analysis return output path: None
Log output:
 ERROR: ERR-002: Limit File Not Found
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 792, in analyze_files
    setpoint_df = pd.read_excel(setpoint_path)
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 481, in read_excel
    io = ExcelFile(
        io,
    ...<2 lines>...
        engine_kwargs=engine_kwargs,
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1604, in __init__
    ext = inspect_excel_format(
        content_or_path=path_or_buffer, storage_options=storage_options
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1452, in inspect_excel_format
    with get_handle(
         ~~~~~~~~~~^
        content_or_path, "rb", storage_options=storage_options, is_text=False
     

### ERR-003: Invalid Configuration DataType (Non-Numeric)

In [15]:
print("--- Executing ERR-003: Non-numeric limit checks ---")

err_limit_dir = os.path.join(test_workspace, 'case_err003')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err003.xlsx')

# Load standard limit and inject non-numeric
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit['Temperature_Limit'] = df_limit['Temperature_Limit'].astype(object)
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Temperature_Limit'] = "NonNumericLimit"
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
assert "ERR-003" in logs, "Logs should capture the non-numeric limit GxP warning"


--- Executing ERR-003: Non-numeric limit checks ---
Log output:
 
 DATE: 2026-05-13

[1/1] Processing Room: 1-P040
ROOM ERROR [1-P040]: ERR-003: Invalid Configuration - High Limit must be a number
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 551, in _analyze_single_room_core
    T_lim = float(setpoint_row['Temperature_Limit'].iloc[0]) if not pd.isna(setpoint_row['Temperature_Limit'].iloc[0]) else 100
            ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'NonNumericLimit'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 996, in analyze_files
    res = _analyze_single_room_core(
        df,
    ...<9 lines>...
        day_analysis_end
    )
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 553, in _analyze_single_room_core
    raise Va

### ERR-004: Audit Trail Tampering Detection

In [16]:
print("--- Executing ERR-004: Audit log tampering ---")

log_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log"
backup_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log.tmp_bak"

# 1. Backup log
if os.path.exists(log_file):
    shutil.copyfile(log_file, backup_file)
    
    # 2. Tamper log
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write('{"timestamp": "2026-06-04 10:00:00", "user": "attacker", "action": "TAMPER", "prev_hash": "wrong", "entry_hash": "invalid"}\n')

    try:
        valid, msg = audit_trail.verify_audit_trail()
        print(f"Verification status: {valid} | Msg: {msg}")
        assert not valid, "Validation must detect tampering"
        print("PASS: Tampering successfully detected.")
    finally:
        # 3. Restore backup
        shutil.copyfile(backup_file, log_file)
        os.remove(backup_file)
        print("Restored audit log backup.")


--- Executing ERR-004: Audit log tampering ---
Verification status: False | Msg: Broken chain at line 9: Hash mismatch.
PASS: Tampering successfully detected.
Restored audit log backup.


### ERR-005: Missing Parameter Raw Data or Columns

In [17]:
print("--- Executing ERR-005: Upfront missing raw files & columns validation ---")

# Test Phase I Missing raw file
out1, logs1, plot1 = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P032"],  # Exists in limit but no CSV in folder
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Phase I Missing file output logs:\n", logs1)
assert out1 is None
assert "ERR-005" in logs1

# Test Phase II Missing sensor files
dummy_scan_dir = os.path.join(test_workspace, 'dummy_p2')
os.makedirs(os.path.join(dummy_scan_dir, '1-P040'), exist_ok=True)
# Only humidity file exists, temp (RMT) is missing
with open(os.path.join(dummy_scan_dir, '1-P040', '1-P040_RMH_2026-05-30_Mock.csv'), 'w') as f:
    f.write("DateTime;Data Source;Value\n2026-05-30 00:00:00;Room Hum;45.2\n")

out2, logs2, plot2 = analysis_logic.analyze_files_phase2(
    folder_path=dummy_scan_dir,
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(dummy_scan_dir)
print("Phase II Missing sensor file logs:\n", logs2)
assert out2 is None
assert "ERR-005" in logs2


--- Executing ERR-005: Upfront missing raw files & columns validation ---
Phase I Missing file output logs:
 ERROR: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 861, in analyze_files
    raise ValueError(f"ERR-005: Raw data file not found in {folder_path} for Room {room_id}")
ValueError: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032

Phase II Missing sensor file logs:
 FILE ERROR [1-P040]: ERR-005: No Temperature file (_RMT_) found in D:\ex_work\AirQualityReview_Project\data\validation_tests\dummy_p2 for 1-P040
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 1329, in analyze_files_phase2
    _, df, sensors = prepare_df_phase2(raw_data_path, room_id=room_id, setpoint_df=setpoint_df)
                     ~~~~~~~~~~~

### ERR-006: Inverted Limits Configuration (High < Low)

In [18]:
print("--- Executing ERR-006: Logical constraint check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err006')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err006.xlsx')

# Create limit file with Low Limit > High Limit
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_High_Limit'] = 20
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_Low_Limit'] = 50
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
assert "ERR-006" in logs, "Logs should capture GxP configuration limit inversion warning"


--- Executing ERR-006: Logical constraint check ---
Log output:
 
 DATE: 2026-05-13

[1/1] Processing Room: 1-P040
ROOM ERROR [1-P040]: ERR-006: Configuration Error - High Limit cannot be lower than Low Limit.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 996, in analyze_files
    res = _analyze_single_room_core(
        df,
    ...<9 lines>...
        day_analysis_end
    )
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 568, in _analyze_single_room_core
    if "ERR-006" in str(e): raise e
                            ^^^^^^^
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 564, in _analyze_single_room_core
    raise ValueError("ERR-006: Configuration Error - High Limit cannot be lower than Low Limit.")
ValueError: ERR-006: Configuration Error - High Limit cannot be lower than Low Limit.



### ERR-007: Excel Report Generation Failure

In [19]:
print("--- Executing ERR-007: Simulate Report Write Lock ---")

# We mock standard ExcelWriter to raise PermissionError and verify ERR-007
import unittest.mock as mock

with mock.patch('pandas.ExcelWriter', side_effect=Exception("Write Lock Permission Denied")):
    out, logs, plot = analysis_logic.analyze_files(
        folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
        setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
        selected_rooms=["1-P040"],
        start_date="2026-05-13",
        end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is None
assert "ERR-007" in logs


--- Executing ERR-007: Simulate Report Write Lock ---
Log output:
 
 DATE: 2026-05-13

[1/1] Processing Room: 1-P040
[1/1] Completed Room: 1-P040
--------------------
ERROR: ERR-007: Report Generation Failed - Write Lock Permission Denied
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 1041, in analyze_files
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
         ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\unittest\mock.py", line 1175, in __call__
    return self._mock_call(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\unittest\mock.py", line 1179, in _mock_call
    return self._execute_mock_call(*args, **kwargs)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\unittest\mock.py", line 1234, in _execute_mock_call
    raise effect
Exception: Write Lock Permission Denied

During handling of the above exception, anot

### ERR-008: Identical Duplicate Timestamps

In [20]:
print("--- Executing ERR-008: Duplicate Timestamp Deduplication ---")

err_dir = os.path.join(test_workspace, 'case_err008')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file with identical duplicate timestamps
csv_dup = (
    '"Key            Name:Suffix                                Trend Definitions Used"\n'
    '"Point_1:","1-P040 ROOM TEMP","","5 minutes"\n'
    '"Point_2:","1-P040 ROOM HUM","","5 minutes"\n'
    '"Point_3:","1-P040 ROOM PRES","","5 minutes"\n'
    '"<>Date","Time","Point_1","Point_2","Point_3"\n'
    '"05/30/2026","00:00","22.5","45.0","40.0"\n'
    '"05/30/2026","00:00","23.0","45.0","40.0"\n'
)
with open(err_file, 'w') as f:
    f.write(csv_dup)

df = analysis_logic.prepare_df(err_file, "1-P040")
print("Deduplicated DataFrame size:", len(df))
print("DataFrame contents:\n", df)
assert len(df) == 1, "Must drop duplicate timestamps and keep only the first record"


--- Executing ERR-008: Duplicate Timestamp Deduplication ---
[WARNING] ERR-008: Duplicate Timestamps Detected in file 1-P040_05-30-26_00-00.csv (Room 1-P040). Found 1 duplicate timestamps from 2026-05-30 00:00:00 to 2026-05-30 00:00:00. Dropping duplicates and keeping the first record.
Deduplicated DataFrame size: 1
DataFrame contents:
 0   DateTime  Temperature  Humidity  Pressure
0 2026-05-30         22.5      45.0      40.0


### ERR-009: Invalid Limit File Structure (Missing Columns)

In [21]:
print("--- Executing ERR-009: Missing columns in limits check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err009')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err009.xlsx')

# Drop a required column
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit = df_limit.drop(columns=['Temperature_Limit'])
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is None
assert "ERR-009" in logs


--- Executing ERR-009: Missing columns in limits check ---
Log output:
 ERROR: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 799, in analyze_files
    raise ValueError(f"ERR-009: Invalid Limit File Format - Missing required columns: {', '.join(missing_cols)}")
ValueError: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit



### ERR-010: Cross-Uploading Data Screens

In [22]:
print("--- Executing ERR-010: Cross-upload checks ---")

# Set up a temp folder with only Phase II files in it
err010_dir = os.path.join(test_workspace, 'case_err010')
os.makedirs(os.path.join(err010_dir, '1-P040'), exist_ok=True)
with open(os.path.join(err010_dir, '1-P040', '1-P040_RMT_2026-05-30.csv'), 'w') as f:
    f.write("DateTime;Value\n2026-05-30 00:00:00;22.5\n")

# Try to feed Phase II directories folder to Phase I analyze_files
out, logs, plot = analysis_logic.analyze_files(
    folder_path=err010_dir,  # Contains Phase II structure
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(err010_dir)
print("Log output:\n", logs)
assert out is None
assert "ERR-010" in logs or "ERR-005" in logs or "No Matching Files" in logs, "Must trigger validation/matching failure"


--- Executing ERR-010: Cross-upload checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-000: Data Loss Remark

In [23]:
print("--- Executing ERR-000: Data Loss Flagging ---")

df_data = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=6, freq='5min'),
    'Temperature': [22.0, 22.1, np.nan, 22.0, 22.3, 22.1], # NaN value introduced
    'Humidity': [45.0, 45.2, 45.1, 45.0, 45.3, 45.1],
    'Pressure': [40.0, 40.2, 40.1, 40.0, 40.3, 40.1]
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

spec_txt, res_txt = analysis_logic._analyze_single_room_core(
    df_data, "1-P040", setpoint_row, "Passed", "Out of spec",
    set(), {}, ['1-P040'], setpoint_row,
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:25:00")
)
print("Analysis Results text:\n", res_txt)
assert "Data Loss" in res_txt, "Must append Data Loss warning tag"


--- Executing ERR-000: Data Loss Flagging ---
Temperature Data Loss Found for 1-P040:
           DateTime Temperature
2026-05-30 00:10:00   Data Loss
-------------------------------
Analysis Results text:
 Temperature: Data Loss
00:10 to 00:10
Humidity: Passed
Pressure: Passed


# Integration / System Logic Verification

### InT-01: Corridor Pressure Comparison and Tolerances

In [24]:
print("--- Executing InT-01: Corridor Pressure checks ---")

corridor_df = pd.DataFrame({
    'DateTime': [pd.Timestamp("2026-05-30 00:00:00")],
    'Pressure': [38.0]
})

dependent_df = pd.DataFrame({
    'DateTime': [pd.Timestamp("2026-05-30 00:00:15")], # 15s offset (within 60s tolerance)
    'Pressure': [35.0]  # Drops below corridor pressure (38.0) -> OVER violation
})

setpoint_df = pd.DataFrame({
    'Room_number': ['1-P040', '1-P036'],
    'Room_Pressure_Comparison': ['1-P036', np.nan], # 1-P040 compares to corridor 1-P036
    'Pressure_Low_Limit': [35.0, 30.0]
})

cache = {'1-P036': corridor_df, '1-P040': dependent_df}

violations = analysis_logic.check_reverse_violations(
    "1-P036", corridor_df, 0, 0, setpoint_df, ['1-P040'], cache
)
print("Detected reverse violations:", violations)
assert len(violations) > 0
assert "over 1-P040" in violations[0]


--- Executing InT-01: Corridor Pressure checks ---
      REVERSE OVER violation data relative to 1-P040:
  DateTime  Pressure_1-P036  Pressure_1-P040  Diff
2026-05-30             38.0             35.0   3.0

Detected reverse violations: ['\n  - 00:00 to 00:00 over 1-P040']


### InT-02: Chart Interval Extraction Filter

In [25]:
print("--- Executing InT-02: Chart Interval min_length=6 filter ---")

# Create 5 rows (duration < 6 continuous rows)
df_short = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=5, freq='5min'),
    'Temperature': [26.0]*5, # Out of spec limit (>25.0)
    'Humidity': [45.0]*5,
    'Pressure': [40.0]*5
})

setpoint_df = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

res = analysis_logic._compute_plot_result(
    {'1-P040': [df_short]},
    setpoint_df, ['1-P040'],
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:20:00")
)
print("Chart summary violations:", res['summary'])
# Expected 0 because length is only 5 rows, and get_intervals requires min_length=6
assert res['summary'][0]['temp_v'] == 0


--- Executing InT-02: Chart Interval min_length=6 filter ---
Chart summary violations: [{'room_id': '1-P040', 'room_name': 'Test Room', 'temp_v': 0, 'hum_v': 0, 'press_v': 0}]


### InT-03: Plot Data Directory Scan

In [26]:
print("--- Executing InT-03: Plot Info gathering ---")

res = analysis_logic.get_plot_info(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13",
    limits=None
)
print("Summary of scan findings:", res.keys())
assert 'summary' in res
assert 'plot_data' in res


--- Executing InT-03: Plot Info gathering ---
Summary of scan findings: dict_keys(['summary', 'plot_data', 'violation_intervals'])


### InT-04: Parameter Violation 25-Min Rule

In [27]:
print("--- Executing InT-04: 25-Minute continuous rule ---")

# Create 6 rows (25-min interval span from row 0 to row 5)
df_long = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=6, freq='5min'),
    'Temperature': [26.0]*6, # Out of spec limit (>25.0) for 25 continuous mins
    'Humidity': [45.0]*6,
    'Pressure': [40.0]*6
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

spec_txt, res_txt = analysis_logic._analyze_single_room_core(
    df_long, "1-P040", setpoint_row, "Passed", "Out of spec",
    set(), {}, ['1-P040'], setpoint_row,
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:25:00")
)
print("Analysis results output text:\n", res_txt)
assert "Out of spec" in res_txt, "Must report Out of spec violation for >= 25 mins"


--- Executing InT-04: 25-Minute continuous rule ---
Temperature Violations Found for 1-P040: (Limit: ≤ 25.0 °C)
           DateTime  Temperature
2026-05-30 00:00:00         26.0
2026-05-30 00:05:00         26.0
2026-05-30 00:10:00         26.0
2026-05-30 00:15:00         26.0
2026-05-30 00:20:00         26.0
2026-05-30 00:25:00         26.0
-------------------------------
Analysis results output text:
 Temperature: Out of spec
00:00 to 00:25 (26.0 to 26.0 °C)
Humidity: Passed
Pressure: Passed


### InT-05: Phase I Full Statistical Execution

In [28]:
print("--- Executing InT-05: Phase I excel report output ---")

out_path, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Output Path generated:", out_path)
assert out_path is not None
assert os.path.exists(out_path)
print("PASS: Excel report successfully verified.")


--- Executing InT-05: Phase I excel report output ---
Output Path generated: reports\AQR_Report_20260608_090633.xlsx
PASS: Excel report successfully verified.


### InT-06: Phase II Full Statistical Execution

In [29]:
print("--- Executing InT-06: Phase II excel report output ---")

out_path, logs, plot = analysis_logic.analyze_files_phase2(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_b11",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit_B11.xlsx",
    selected_rooms=["11-1-012", "11-1-013"],
    start_date="2026-05-01",
    end_date="2026-05-05"
)
print("Output Path generated:", out_path)
assert out_path is not None
assert os.path.exists(out_path)
print("PASS: Phase II Excel report successfully verified.")


--- Executing InT-06: Phase II excel report output ---
Output Path generated: reports\AQR_Report_P2_20260608_090634.xlsx
PASS: Phase II Excel report successfully verified.
